# Trailwave Freight: ETL pipeline

Loads the DataCo Smart Supply Chain CSV into a PostgreSQL star schema.

1. **Extract**: read the CSV (only the columns the report needs).
2. **Transform**: clean text, translate country names to English, build five dimensions and one fact table at order item grain.
3. **Check**: assert the transformed data matches the source before anything is written.
4. **Load**: recreate the schema and insert everything in one transaction.
5. **Verify**: compare totals in PostgreSQL with the totals in pandas.

Source: DataCo Smart Supply Chain for Big Data Analysis (Pereira, 2019), Mendeley Data, CC BY 4.0.
There is no need to run `sql/01_create_schema.sql` separately: this notebook runs it.

In [ ]:
import os
from pathlib import Path

import pandas as pd
import psycopg2
from dotenv import load_dotenv
from psycopg2.extras import execute_values


def find_project_root(start):
    # Walk up from the current folder until the project folder is found,
    # so the notebook works whether Jupyter starts in python/ or in the project root.
    for folder in [start, *start.parents]:
        if (folder / "sql" / "01_create_schema.sql").is_file():
            return folder
    raise FileNotFoundError(
        "Project folder not found. Open the notebook from inside the project "
        "(the folder that contains sql/01_create_schema.sql)."
    )


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
SCHEMA_PATH = PROJECT_ROOT / "sql" / "01_create_schema.sql"
ENV_PATH = PROJECT_ROOT / ".env"

# Credentials live in a .env file at the project root (never committed).
# override=True: values in .env win over anything already loaded in this Jupyter session.
load_dotenv(ENV_PATH, override=True)

# The CSV is looked for in this order:
#   1. DATA_PATH in .env, if set
#   2. data/ inside the project
#   3. data/ in the folder that contains the project
CSV_NAME = "DataCoSupplyChainDataset.csv"
candidates = [
    os.getenv("DATA_PATH"),
    PROJECT_ROOT / "data" / CSV_NAME,
    PROJECT_ROOT.parent / "data" / CSV_NAME,
]
DATA_PATH = next((Path(c) for c in candidates if c and Path(c).is_file()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        f"{CSV_NAME} not found. Copy it to {PROJECT_ROOT / 'data'} "
        "or set DATA_PATH in .env. See data/README.md for the download link."
    )

DB_CONFIG = {
    "host": os.getenv("DB_HOST", "localhost"),
    "port": os.getenv("DB_PORT", "5432"),
    "dbname": os.getenv("DB_NAME"),
    "user": os.getenv("DB_USER"),
    "password": os.getenv("DB_PASSWORD"),
}
# Drop empty values so psycopg2 falls back to its defaults.
DB_CONFIG = {key: value for key, value in DB_CONFIG.items() if value}

print(f"Project folder: {PROJECT_ROOT.name}")
print(f"CSV found:      {DATA_PATH.name} ({DATA_PATH.stat().st_size / 1e6:.0f} MB)")
print(f".env found:     {ENV_PATH.is_file()}")
print(
    f"Database:       {DB_CONFIG.get('dbname', '(not set)')} on {DB_CONFIG['host']}:{DB_CONFIG['port']}"
)
print(
    f"DB user:        {DB_CONFIG.get('user', '(not set)')} | password set: {'password' in DB_CONFIG}"
)

CALENDAR_START = "2015-01-01"
CALENDAR_END = "2018-12-31"

Project folder: final_version
CSV found:      DataCoSupplyChainDataset.csv (96 MB)
.env found:     True
Database:       trailwave on localhost:5433
DB user:        postgres | password set: True


## 1. Extract

In [ ]:
USE_COLS = [
    "Order Item Id",
    "Order Id",
    "order date (DateOrders)",
    "shipping date (DateOrders)",
    "Customer Id",
    "Customer Segment",
    "Customer City",
    "Customer State",
    "Customer Country",
    "Product Card Id",
    "Product Name",
    "Product Price",
    "Category Id",
    "Category Name",
    "Department Id",
    "Department Name",
    "Market",
    "Order Region",
    "Order Country",
    "Shipping Mode",
    "Days for shipping (real)",
    "Days for shipment (scheduled)",
    "Order Status",
    "Delivery Status",
    "Late_delivery_risk",
    "Order Item Quantity",
    "Order Item Product Price",
    "Order Item Discount",
    "Order Item Discount Rate",
    "Sales",
    "Order Item Total",
    "Order Profit Per Order",
]

# The file is encoded as latin-1.
raw = pd.read_csv(DATA_PATH, encoding="latin-1", usecols=USE_COLS)
print(f"Rows: {len(raw):,} | Columns: {raw.shape[1]}")
raw.head(3)

Rows: 180,519 | Columns: 32


,Days for shipping (real),Days for shipment (scheduled),Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,Customer Country,Customer Id,Customer Segment,...,Sales,Order Item Total,Order Profit Per Order,Order Region,Order Status,Product Card Id,Product Name,Product Price,shipping date (DateOrders),Shipping Mode
0,3,4,Advance shipping,0,73,Sporting Goods,Caguas,Puerto Rico,20755,Consumer,...,327.75,314.640015,91.250000,Southeast Asia,COMPLETE,1360,Smart watch,327.75,2/3/2018 22:56,Standard Class
1,5,4,Late delivery,1,73,Sporting Goods,Caguas,Puerto Rico,19492,Consumer,...,327.75,311.359985,-249.089996,South Asia,PENDING,1360,Smart watch,327.75,1/18/2018 12:27,Standard Class
2,4,4,Shipping on time,0,73,Sporting Goods,San Jose,EE. UU.,19491,Consumer,...,327.75,309.720001,-247.779999,South Asia,CLOSED,1360,Smart watch,327.75,1/17/2018 12:06,Standard Class


## 2. Transform

### 2.1 Clean text

Some text values carry trailing or doubled spaces (for example `"South of  USA "` and `"Health and Beauty "`).

In [3]:
text_cols = [c for c in raw.columns if pd.api.types.is_string_dtype(raw[c])]
for col in text_cols:
    raw[col] = raw[col].str.replace(r"\s+", " ", regex=True).str.strip()

print(sorted(raw["Order Region"].unique()))

['Canada', 'Caribbean', 'Central Africa', 'Central America', 'Central Asia', 'East Africa', 'East of USA', 'Eastern Asia', 'Eastern Europe', 'North Africa', 'Northern Europe', 'Oceania', 'South America', 'South Asia', 'South of USA', 'Southeast Asia', 'Southern Africa', 'Southern Europe', 'US Center', 'West Africa', 'West Asia', 'West of USA', 'Western Europe']


### 2.2 Translate country names

Order countries come in Spanish (for example `Japón`, `Alemania`). The report is in English, and a map needs
recognised English names, so every country is translated through an explicit lookup.
The cell fails if a country has no translation, so nothing slips through unnoticed.

In [ ]:
COUNTRY_EN = {
    "Afganistán": "Afghanistan",
    "Albania": "Albania",
    "Alemania": "Germany",
    "Angola": "Angola",
    "Arabia Saudí": "Saudi Arabia",
    "Argelia": "Algeria",
    "Argentina": "Argentina",
    "Armenia": "Armenia",
    "Australia": "Australia",
    "Austria": "Austria",
    "Azerbaiyán": "Azerbaijan",
    "Bangladés": "Bangladesh",
    "Barbados": "Barbados",
    "Baréin": "Bahrain",
    "Belice": "Belize",
    "Benín": "Benin",
    "Bielorrusia": "Belarus",
    "Bolivia": "Bolivia",
    "Bosnia y Herzegovina": "Bosnia and Herzegovina",
    "Botsuana": "Botswana",
    "Brasil": "Brazil",
    "Bulgaria": "Bulgaria",
    "Burkina Faso": "Burkina Faso",
    "Burundi": "Burundi",
    "Bután": "Bhutan",
    "Bélgica": "Belgium",
    "Camboya": "Cambodia",
    "Camerún": "Cameroon",
    "Canada": "Canada",
    "Chad": "Chad",
    "Chile": "Chile",
    "China": "China",
    "Chipre": "Cyprus",
    "Colombia": "Colombia",
    "Corea del Sur": "South Korea",
    "Costa Rica": "Costa Rica",
    "Costa de Marfil": "Ivory Coast",
    "Croacia": "Croatia",
    "Cuba": "Cuba",
    "Dinamarca": "Denmark",
    "Ecuador": "Ecuador",
    "Egipto": "Egypt",
    "El Salvador": "El Salvador",
    "Emiratos Árabes Unidos": "United Arab Emirates",
    "Eritrea": "Eritrea",
    "Eslovaquia": "Slovakia",
    "Eslovenia": "Slovenia",
    "España": "Spain",
    "Estados Unidos": "United States",
    "Estonia": "Estonia",
    "Etiopía": "Ethiopia",
    "Filipinas": "Philippines",
    "Finlandia": "Finland",
    "Francia": "France",
    "Gabón": "Gabon",
    "Georgia": "Georgia",
    "Ghana": "Ghana",
    "Grecia": "Greece",
    "Guadalupe": "Guadeloupe",
    "Guatemala": "Guatemala",
    "Guayana Francesa": "French Guiana",
    "Guinea": "Guinea",
    "Guinea Ecuatorial": "Equatorial Guinea",
    "Guinea-Bissau": "Guinea-Bissau",
    "Guyana": "Guyana",
    "Haití": "Haiti",
    "Honduras": "Honduras",
    "Hong Kong": "Hong Kong",
    "Hungría": "Hungary",
    "India": "India",
    "Indonesia": "Indonesia",
    "Irak": "Iraq",
    "Irlanda": "Ireland",
    "Irán": "Iran",
    "Israel": "Israel",
    "Italia": "Italy",
    "Jamaica": "Jamaica",
    "Japón": "Japan",
    "Jordania": "Jordan",
    "Kazajistán": "Kazakhstan",
    "Kenia": "Kenya",
    "Kirguistán": "Kyrgyzstan",
    "Kuwait": "Kuwait",
    "Laos": "Laos",
    "Lesoto": "Lesotho",
    "Liberia": "Liberia",
    "Libia": "Libya",
    "Lituania": "Lithuania",
    "Luxemburgo": "Luxembourg",
    "Líbano": "Lebanon",
    "Macedonia": "North Macedonia",
    "Madagascar": "Madagascar",
    "Malasia": "Malaysia",
    "Mali": "Mali",
    "Marruecos": "Morocco",
    "Martinica": "Martinique",
    "Mauritania": "Mauritania",
    "Moldavia": "Moldova",
    "Mongolia": "Mongolia",
    "Montenegro": "Montenegro",
    "Mozambique": "Mozambique",
    "Myanmar (Birmania)": "Myanmar",
    "México": "Mexico",
    "Namibia": "Namibia",
    "Nepal": "Nepal",
    "Nicaragua": "Nicaragua",
    "Nigeria": "Nigeria",
    "Noruega": "Norway",
    "Nueva Zelanda": "New Zealand",
    "Níger": "Niger",
    "Omán": "Oman",
    "Pakistán": "Pakistan",
    "Panamá": "Panama",
    "Papúa Nueva Guinea": "Papua New Guinea",
    "Paraguay": "Paraguay",
    "Países Bajos": "Netherlands",
    "Perú": "Peru",
    "Polonia": "Poland",
    "Portugal": "Portugal",
    "Qatar": "Qatar",
    "Reino Unido": "United Kingdom",
    "República Centroafricana": "Central African Republic",
    "República Checa": "Czechia",
    "República Democrática del Congo": "Democratic Republic of the Congo",
    "República Dominicana": "Dominican Republic",
    "República de Gambia": "Gambia",
    "República del Congo": "Republic of the Congo",
    "Ruanda": "Rwanda",
    "Rumania": "Romania",
    "Rusia": "Russia",
    "Senegal": "Senegal",
    "Serbia": "Serbia",
    "Sierra Leona": "Sierra Leone",
    "Singapur": "Singapore",
    "Siria": "Syria",
    "Somalia": "Somalia",
    "Sri Lanka": "Sri Lanka",
    "Suazilandia": "Eswatini",
    "SudAfrica": "South Africa",
    "Sudán": "Sudan",
    "Sudán del Sur": "South Sudan",
    "Suecia": "Sweden",
    "Suiza": "Switzerland",
    "Surinam": "Suriname",
    "Sáhara Occidental": "Western Sahara",
    "Tailandia": "Thailand",
    "Taiwán": "Taiwan",
    "Tanzania": "Tanzania",
    "Tayikistán": "Tajikistan",
    "Togo": "Togo",
    "Trinidad y Tobago": "Trinidad and Tobago",
    "Turkmenistán": "Turkmenistan",
    "Turquía": "Turkey",
    "Túnez": "Tunisia",
    "Ucrania": "Ukraine",
    "Uganda": "Uganda",
    "Uruguay": "Uruguay",
    "Uzbekistán": "Uzbekistan",
    "Venezuela": "Venezuela",
    "Vietnam": "Vietnam",
    "Yemen": "Yemen",
    "Yibuti": "Djibouti",
    "Zambia": "Zambia",
    "Zimbabue": "Zimbabwe",
}

missing = sorted(set(raw["Order Country"]) - set(COUNTRY_EN))
assert not missing, f"Countries without a translation: {missing}"

raw["Order Country"] = raw["Order Country"].map(COUNTRY_EN)

# Customer country only has two values in the source.
raw["Customer Country"] = raw["Customer Country"].replace({"EE. UU.": "United States"})
print(
    raw["Customer Country"].unique(),
    "|",
    raw["Order Country"].nunique(),
    "order countries",
)

<ArrowStringArray>
['Puerto Rico', 'United States']
Length: 2, dtype: str | 164 order countries


### 2.3 Dates

Order and shipping timestamps are reduced to calendar days. `date_id` is the integer `YYYYMMDD`.
The calendar covers four full years (2015 to 2018) so time intelligence works on complete years.
2018 only has orders for January.

In [ ]:
order_dt = pd.to_datetime(raw["order date (DateOrders)"])
ship_dt = pd.to_datetime(raw["shipping date (DateOrders)"])

raw["order_date_id"] = order_dt.dt.strftime("%Y%m%d").astype(int)
raw["ship_date_id"] = ship_dt.dt.strftime("%Y%m%d").astype(int)

days = pd.date_range(CALENDAR_START, CALENDAR_END, freq="D")
dim_date = pd.DataFrame({"full_date": days.date})
dim_date["date_id"] = days.strftime("%Y%m%d").astype(int)
dim_date["year"] = days.year
dim_date["quarter"] = days.quarter
dim_date["month"] = days.month
dim_date["month_name"] = days.strftime("%B")
dim_date["year_month"] = days.strftime("%Y-%m")
dim_date["month_start"] = days.to_period("M").to_timestamp().date
dim_date["week_of_year"] = days.isocalendar().week.to_numpy()
dim_date["day_of_week"] = days.dayofweek + 1
dim_date["day_name"] = days.strftime("%A")
dim_date["is_weekend"] = days.dayofweek >= 5
dim_date = dim_date[
    [
        "date_id",
        "full_date",
        "year",
        "quarter",
        "month",
        "month_name",
        "year_month",
        "month_start",
        "week_of_year",
        "day_of_week",
        "day_name",
        "is_weekend",
    ]
]

covered = set(dim_date["date_id"])
assert set(raw["order_date_id"]) <= covered, "Order dates outside the calendar"
assert set(raw["ship_date_id"]) <= covered, "Ship dates outside the calendar"
print(
    f"dim_date: {len(dim_date):,} days | orders from {order_dt.min().date()} to {order_dt.max().date()}"
)

dim_date: 1,461 days | orders from 2015-01-01 to 2018-01-31


### 2.4 Dimensions

In [ ]:
# Shipping mode: promised days are constant per mode (checked below).
dim_shipping_mode = (
    raw.groupby("Shipping Mode", as_index=False)["Days for shipment (scheduled)"]
    .agg(["min", "max"])
    .reset_index()
)
assert (dim_shipping_mode["min"] == dim_shipping_mode["max"]).all(), (
    "Promised days vary inside a mode"
)

dim_shipping_mode = (
    dim_shipping_mode.rename(
        columns={"Shipping Mode": "shipping_mode", "min": "promised_days"}
    )[["shipping_mode", "promised_days"]]
    .sort_values(["promised_days", "shipping_mode"])
    .reset_index(drop=True)
)
dim_shipping_mode.insert(0, "shipping_mode_id", dim_shipping_mode.index + 1)
dim_shipping_mode

,shipping_mode_id,shipping_mode,promised_days
0,1,Same Day,0
1,2,First Class,1
2,3,Second Class,2
3,4,Standard Class,4


In [ ]:
dim_geography = (
    raw[["Market", "Order Region", "Order Country"]]
    .drop_duplicates()
    .rename(
        columns={
            "Market": "market",
            "Order Region": "region",
            "Order Country": "country",
        }
    )
    .sort_values(["market", "region", "country"])
    .reset_index(drop=True)
)
dim_geography.insert(0, "geography_id", dim_geography.index + 1)

dim_product = (
    raw[
        [
            "Product Card Id",
            "Product Name",
            "Category Id",
            "Category Name",
            "Department Id",
            "Department Name",
            "Product Price",
        ]
    ]
    .drop_duplicates()
    .rename(
        columns={
            "Product Card Id": "product_id",
            "Product Name": "product_name",
            "Category Id": "category_id",
            "Category Name": "category_name",
            "Department Id": "department_id",
            "Department Name": "department_name",
            "Product Price": "unit_price",
        }
    )
    .sort_values("product_id")
    .reset_index(drop=True)
)
assert dim_product["product_id"].is_unique, "A product has more than one description"

dim_customer = (
    raw[
        [
            "Customer Id",
            "Customer Segment",
            "Customer City",
            "Customer State",
            "Customer Country",
        ]
    ]
    .drop_duplicates()
    .rename(
        columns={
            "Customer Id": "customer_id",
            "Customer Segment": "customer_segment",
            "Customer City": "customer_city",
            "Customer State": "customer_state",
            "Customer Country": "customer_country",
        }
    )
    .sort_values("customer_id")
    .reset_index(drop=True)
)
assert dim_customer["customer_id"].is_unique, "A customer has more than one description"

for name, frame in [
    ("dim_geography", dim_geography),
    ("dim_product", dim_product),
    ("dim_customer", dim_customer),
]:
    print(f"{name}: {len(frame):,} rows")

dim_geography: 167 rows
dim_product: 118 rows
dim_customer: 20,652 rows


### 2.5 Fact table (order item grain)

- `is_canceled`: delivery status is `Shipping canceled` (orders with status CANCELED or SUSPECTED_FRAUD). Nothing was delivered, so delivery KPIs leave these out.
- `is_late`: the source flag `Late_delivery_risk` (it matches `Delivery Status = 'Late delivery'` exactly).
- `days_late`: real days minus promised days, never below zero; empty for canceled items.

In [ ]:
fact = raw.merge(
    dim_geography.rename(
        columns={
            "market": "Market",
            "region": "Order Region",
            "country": "Order Country",
        }
    ),
    on=["Market", "Order Region", "Order Country"],
    how="left",
).merge(
    dim_shipping_mode[["shipping_mode_id", "shipping_mode"]].rename(
        columns={"shipping_mode": "Shipping Mode"}
    ),
    on="Shipping Mode",
    how="left",
)

fact["is_canceled"] = fact["Delivery Status"].eq("Shipping canceled")
fact["is_late"] = fact["Late_delivery_risk"].eq(1)
delta = fact["Days for shipping (real)"] - fact["Days for shipment (scheduled)"]
fact["days_late"] = delta.clip(lower=0).astype("Int64").mask(fact["is_canceled"])

fact_order_items = fact.rename(
    columns={
        "Order Item Id": "order_item_id",
        "Order Id": "order_id",
        "Customer Id": "customer_id",
        "Product Card Id": "product_id",
        "Order Status": "order_status",
        "Delivery Status": "delivery_status",
        "Days for shipping (real)": "days_real",
        "Days for shipment (scheduled)": "days_scheduled",
        "Order Item Quantity": "quantity",
        "Order Item Product Price": "unit_price",
        "Order Item Discount": "discount_amount",
        "Order Item Discount Rate": "discount_rate",
        "Sales": "sales",
        "Order Item Total": "item_total",
        "Order Profit Per Order": "profit",
    }
)[
    [
        "order_item_id",
        "order_id",
        "order_date_id",
        "ship_date_id",
        "customer_id",
        "product_id",
        "geography_id",
        "shipping_mode_id",
        "order_status",
        "delivery_status",
        "is_canceled",
        "is_late",
        "days_real",
        "days_scheduled",
        "days_late",
        "quantity",
        "unit_price",
        "discount_amount",
        "discount_rate",
        "sales",
        "item_total",
        "profit",
    ]
]
# Money columns carry floating point noise in the source (for example 299.980011).
# They are rounded to cents, which is how they are stored in PostgreSQL (NUMERIC with 2 decimals).
money_cols = ["unit_price", "discount_amount", "sales", "item_total", "profit"]
fact_order_items[money_cols] = fact_order_items[money_cols].round(2)
fact_order_items["discount_rate"] = fact_order_items["discount_rate"].round(4)
dim_product["unit_price"] = dim_product["unit_price"].round(2)

print(f"fact_order_items: {len(fact_order_items):,} rows")
fact_order_items.head(3)

fact_order_items: 180,519 rows


,order_item_id,order_id,order_date_id,ship_date_id,customer_id,product_id,geography_id,shipping_mode_id,order_status,delivery_status,...,days_real,days_scheduled,days_late,quantity,unit_price,discount_amount,discount_rate,sales,item_total,profit
0,180517,77202,20180131,20180203,20755,1360,139,4,COMPLETE,Advance shipping,...,3,4,0,1,327.75,13.11,0.04,327.75,314.64,91.25
1,179254,75939,20180113,20180118,19492,1360,133,4,PENDING,Late delivery,...,5,4,1,1,327.75,16.39,0.05,327.75,311.36,-249.09
2,179253,75938,20180113,20180117,19491,1360,133,4,CLOSED,Shipping on time,...,4,4,0,1,327.75,18.03,0.06,327.75,309.72,-247.78


## 3. Check before loading

Every assertion compares the transformed tables with the source. If one fails, nothing is written.

In [ ]:
checks = {
    "same number of rows as the source": len(fact_order_items) == len(raw),
    "order_item_id is unique": fact_order_items["order_item_id"].is_unique,
    "no missing geography or shipping mode": fact_order_items[
        ["geography_id", "shipping_mode_id"]
    ]
    .notna()
    .all()
    .all(),
    "all customers exist in dim_customer": fact_order_items["customer_id"]
    .isin(dim_customer["customer_id"])
    .all(),
    "all products exist in dim_product": fact_order_items["product_id"]
    .isin(dim_product["product_id"])
    .all(),
    # Rounding to cents moves a total by well under one dollar across 180k rows.
    "item_total matches source (within $1)": abs(
        fact_order_items["item_total"].sum() - raw["Order Item Total"].sum()
    )
    < 1,
    "profit matches source (within $1)": abs(
        fact_order_items["profit"].sum() - raw["Order Profit Per Order"].sum()
    )
    < 1,
    "sales matches source (within $1)": abs(
        fact_order_items["sales"].sum() - raw["Sales"].sum()
    )
    < 1,
    "distinct orders match source": fact_order_items["order_id"].nunique()
    == raw["Order Id"].nunique(),
    "late flag matches delivery status": (
        fact_order_items["is_late"]
        == fact_order_items["delivery_status"].eq("Late delivery")
    ).all(),
    "delivery fields are the same on every item of an order": fact_order_items.groupby(
        "order_id"
    )[["shipping_mode_id", "days_real", "is_late", "delivery_status"]]
    .nunique()
    .max()
    .max()
    == 1,
    "canceled items have no days_late": fact_order_items.loc[
        fact_order_items["is_canceled"], "days_late"
    ]
    .isna()
    .all(),
    "late items are never canceled": not (
        fact_order_items["is_late"] & fact_order_items["is_canceled"]
    ).any(),
}

for label, passed in checks.items():
    print(("PASS  " if passed else "FAIL  ") + label)
assert all(checks.values()), "At least one check failed: nothing will be loaded."

PASS  same number of rows as the source
PASS  order_item_id is unique
PASS  no missing geography or shipping mode
PASS  all customers exist in dim_customer
PASS  all products exist in dim_product
PASS  item_total matches source (within $1)
PASS  profit matches source (within $1)
PASS  sales matches source (within $1)
PASS  distinct orders match source
PASS  late flag matches delivery status
PASS  delivery fields are the same on every item of an order
PASS  canceled items have no days_late
PASS  late items are never canceled


## 4. Load

The schema is recreated and every table is filled in one transaction. If anything fails, the transaction
rolls back and the database stays as it was.

In [ ]:
def to_records(frame):
    # Convert a DataFrame to plain Python tuples, turning missing values into None.
    return frame.astype(object).where(frame.notna(), None).values.tolist()


def insert(cur, table, frame, page_size=10_000):
    columns = ", ".join(frame.columns)
    execute_values(
        cur,
        f"INSERT INTO {table} ({columns}) VALUES %s",
        to_records(frame),
        page_size=page_size,
    )
    print(f"  {table}: {len(frame):,} rows")


missing = [key for key in ("DB_NAME", "DB_USER") if not os.getenv(key)]
if missing:
    raise RuntimeError(
        f"Missing in .env: {', '.join(missing)}. Copy .env.example to .env and fill in your values."
    )

conn = psycopg2.connect(**DB_CONFIG)
try:
    with conn:  # commits on success, rolls back on error
        with conn.cursor() as cur:
            cur.execute(SCHEMA_PATH.read_text())
            print("Schema created. Loading:")
            insert(cur, "dim_date", dim_date)
            insert(cur, "dim_product", dim_product)
            insert(cur, "dim_customer", dim_customer)
            insert(cur, "dim_geography", dim_geography)
            insert(cur, "dim_shipping_mode", dim_shipping_mode)
            insert(cur, "fact_order_items", fact_order_items)
    print("Load committed.")
finally:
    conn.close()

Schema created. Loading:
  dim_date: 1,461 rows
  dim_product: 118 rows
  dim_customer: 20,652 rows
  dim_geography: 167 rows
  dim_shipping_mode: 4 rows
  fact_order_items: 180,519 rows
Load committed.


## 5. Verify in PostgreSQL

Totals read back from the database must equal the totals in pandas.

In [ ]:
conn = psycopg2.connect(**DB_CONFIG)
try:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT COUNT(*), COUNT(DISTINCT order_id),
                   ROUND(SUM(sales), 2), ROUND(SUM(item_total), 2), ROUND(SUM(profit), 2),
                   COUNT(*) FILTER (WHERE is_canceled), COUNT(DISTINCT order_id) FILTER (WHERE is_late)
            FROM fact_order_items
        """)
        db = cur.fetchone()
finally:
    conn.close()

expected = (
    len(fact_order_items),
    fact_order_items["order_id"].nunique(),
    round(float(fact_order_items["sales"].sum()), 2),
    round(float(fact_order_items["item_total"].sum()), 2),
    round(float(fact_order_items["profit"].sum()), 2),
    int(fact_order_items["is_canceled"].sum()),
    fact_order_items.loc[fact_order_items["is_late"], "order_id"].nunique(),
)
labels = [
    "items",
    "orders",
    "sales",
    "item_total",
    "profit",
    "canceled items",
    "late orders",
]
summary = pd.DataFrame(
    {"pandas": expected, "postgres": [float(v) for v in db]}, index=labels
)
summary["match"] = (summary["pandas"] - summary["postgres"]).abs() < 0.01
print(summary.to_string())
assert summary["match"].all(), "Database totals differ from pandas"
print("All totals match.")

                     pandas     postgres  match
items             180519.00    180519.00   True
orders             65752.00     65752.00   True
sales           36784734.31  36784734.31   True
item_total      33054402.04  33054402.04   True
profit           3966902.97   3966902.97   True
canceled items      7754.00      7754.00   True
late orders        36048.00     36048.00   True
All totals match.
